In [1]:
!wget https://zhanggroup.org/BioLiP/download/BioLiP_nr.txt.gz -O downloads/biolip.txt.gz
!wget https://zhanggroup.org/BioLiP/data/protein_nr.fasta.gz -O downloads/biolip.fasta.gz

.downloads/biolip.txt.gz: No such file or directory
--2025-09-10 15:06:36--  https://zhanggroup.org/BioLiP/data/protein_nr.fasta.gz
Resolving zhanggroup.org (zhanggroup.org)... 137.132.93.250
Connecting to zhanggroup.org (zhanggroup.org)|137.132.93.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8021454 (7.6M) [application/x-gzip]
Saving to: ‘.downloadsbiolip.fasta.gz’

.downloadsbiolip.fa 100%[===================>]   7.65M   319KB/s    in 24s     

2025-09-10 15:07:01 (321 KB/s) - ‘.downloadsbiolip.fasta.gz’ saved [8021454/8021454]



In [1]:
import pandas as pd
biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)
# reasonable column names for list above
biolip_df.columns = [
    'pdb_id', 'receptor_chain', 'resolution', 'binding_site_number_code',
    'ligand_id', 'ligand_chain', 'ligand_serial_number', 'binding_site_residues_pdb_num',
    'binding_site_residues_renumbered', 'catalytic_site_residues_pdb_num',
    'catalytic_site_residues_renumbered', 'ec_number', 'go_terms',
    'binding_affinity_manual_survey', 'binding_affinity_binding_moad',
    'binding_affinity_pdbbind_cn', 'binding_affinity_bindingdb',
    'uniprot_id', 'pubmed_id', 'ligand_residue_sequence_number',
    'receptor_sequence'
]
biolip_df.dropna(subset=['pdb_id', 'receptor_chain'], inplace=True)
# biolip_df['seq_len'] = biolip_df['receptor_sequence'].apply(len)
biolip_df['fasta_id'] = biolip_df['pdb_id'].str.cat(biolip_df['receptor_chain'].str.strip())

#load biolip fasta file with biopython
from Bio import SeqIO
fasta_sequences = SeqIO.parse(open('downloads/biolip.fasta'), 'fasta')
fasta_dict = {record.id: str(record.seq) for record in fasta_sequences}
biolip_df = biolip_df[biolip_df['fasta_id'].isin(fasta_dict)]
biolip_df['protein_sequence'] = biolip_df['fasta_id'].map(fasta_dict)
biolip_df['seq_len'] = biolip_df['protein_sequence'].apply(len)

/tmp/ipykernel_3146056/838665195.py:2: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)


In [16]:
biolip_df['ligand_id'].value_counts().head(20)
#select 200 random rows from each of the top 10 ligands and save to new dataframe
top_20_ligands = biolip_df['ligand_id'].value_counts().head(9).index.tolist()
biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=120, random_state=1) if len(x) >= 120 else x).reset_index(drop=True)
biolip_top_20 = biolip_top_20[biolip_top_20['seq_len'] < 850]

/tmp/ipykernel_3146056/2486878590.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=120, random_state=1) if len(x) >= 120 else x).reset_index(drop=True)


In [17]:
def get_binding_side_residues(res_renum_str):
    res = res_renum_str.split(' ')
    res_ind = [int(r[1:]) for r in res]
    return res_ind
def get_binding_site_AA(res_renum_str):
    res = res_renum_str.split(' ')
    res_aa = [r[0] for r in res]
    return res_aa
def get_site_AA(res_ind, seq):
    site_aa = [seq[i-1] for i in res_ind if i-1 < len(seq)]
    return site_aa
biolip_top_20['binding_site_residues_list'] = biolip_top_20['binding_site_residues_renumbered'].apply(get_binding_side_residues)
biolip_top_20['binding_site_AA'] = biolip_top_20['binding_site_residues_renumbered'].apply(get_binding_site_AA)
biolip_top_20['binding_site_AA_seq'] = biolip_top_20.apply(lambda row: get_site_AA(row['binding_site_residues_list'], row['protein_sequence']), axis=1)

In [18]:
biolip_top_20['go_terms'] = biolip_top_20['go_terms'].fillna('').apply(lambda x: x.split(',') if x != '' else [])
biolip_top_20['go_terms'] = biolip_top_20['go_terms'].apply(lambda x: ['GO:' + term.zfill(7) if not term.startswith('GO:') else term for term in x])
biolip_top_20 = biolip_top_20[biolip_top_20['go_terms'].map(len) > 0]

In [19]:
columns = ['UniprotID', 'AnnotatedIndices', 'GOTerm', 'Sequence']
col_map = {'uniprot_id': 'UniprotID', 
           'binding_site_residues_list': 'AnnotatedIndices', 
           'go_terms': 'GOTerm', 
           'protein_sequence': 'Sequence', 'ligand_id': 'LigandID',}

In [20]:
biolip_top_20['ligand_id'].unique()

array(['ADP', 'CA', 'CLA', 'MG', 'MN', 'ZN', 'dna', 'peptide', 'rna'],
      dtype=object)

In [21]:
biolip_top_20 = biolip_top_20[[col for col in col_map.keys()]]
biolip_top_20.rename(columns=col_map, inplace=True)
biolip_top_20.reset_index(drop=True, inplace=True)
biolip_top_20.to_csv('datasets/biolip_dataset.csv', index=False, sep='\t')

In [22]:
biolip_top_20

,UniprotID,AnnotatedIndices,GOTerm,Sequence,LigandID
0,P0AEF0,"[66, 74, 75, 108, 109, 110, 111, 112, 113, 236...","[GO:0003677, GO:0005515, GO:0005524, GO:000626...",MKNVGDLMQRLQKMMPAHIKPAFKTGEELLAWQKEQGAIRSAALER...,ADP
1,Q9SCL7,"[31, 201, 202, 207, 210, 235, 237, 238, 241]","[GO:0003991, GO:0005524, GO:0005737, GO:000652...",SPDYRVEILSESLPFIQKFRGKTIVVKYGGAAMTSPELKSSVVSDL...,ADP
2,Q9Y265,"[16, 17, 19, 37, 39, 72, 74, 75, 76, 77, 359, ...","[GO:0000492, GO:0000723, GO:0000786, GO:000081...",KIEEVKSTTKTQRIASHSHVKGLGLDESGLAKQAASGLVGQENARE...,ADP
3,P28738,"[11, 13, 14, 85, 86, 88, 89, 90, 91]","[GO:0003777, GO:0005524, GO:0007018, GO:0008017]",PAECSIKVMCRFRPLNEAEILRGDKFIPKFKGEETVVIGQGKPYVF...,ADP
4,A0A369R1N0,"[174, 191, 237, 238, 239, 317, 323]","[GO:0000166, GO:0003824, GO:0004356, GO:000654...",MTDLASIAREKGIEFFLISFTDLLGVQRAKLVPARAIADMAVNGAG...,ADP
...,...,...,...,...,...
807,P48237,"[35, 74, 124, 129, 130, 131, 134, 138, 167, 17...","[GO:0000002, GO:0000963, GO:0003729, GO:000573...",DILAKNLFDRSHSNWAPVIDRLYVSEKRFMDIDSREFSVWLNGTVK...,rna
808,P75544,"[35, 36, 38, 40, 41, 42, 315, 316, 317, 346, 3...","[GO:0003746, GO:0003924, GO:0005525, GO:000573...",TVDLINFRNFGIMAHIDAGKTTTSERILFHSGRIHKIGETHDGESV...,rna
809,O51140,"[28, 29, 31, 32, 40, 43, 44, 47, 50, 51, 53, 54]","[GO:0003735, GO:0005840, GO:0006412]",SCKFCDSGKHPDYKDFDFLKKFITEQGKILPKRITGTSAKHQRRLA...,rna
810,S7W6N9,"[1, 2, 3, 4, 5, 7, 8, 11, 12, 15, 16, 17, 18, ...","[GO:0003735, GO:0005840, GO:0006412, GO:1990904]",SRKTKKVGITGKYGSRYGSSLKRRAMICMNAQSKRYCCSFCGKTTV...,rna
